In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent 
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver

# 加载环境变量
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# 初始化模型
model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)


In [6]:
# 创建持久化的 checkpointer（使用 with 语句）
db_path = "checkpoints.sqlite"  # 相对路径

with SqliteSaver.from_conn_string(db_path) as checkpointer:
    agent = create_agent(
        model=model,
        tools=[],
        system_prompt="你是一个有帮助的助手。",
        checkpointer=checkpointer  # 使用 SQLite 持久化
    )

    config = {"configurable": {"thread_id": "persistent_session"}}

    print("\n第一轮对话：")
    print("用户: 我叫李四")
    agent.invoke(
        {"messages": [{"role": "user", "content": "我叫李四"}]},
        config=config
    )

    print("\n第二轮对话：")
    print("用户: 我叫什么？")
    response = agent.invoke(
        {"messages": [{"role": "user", "content": "我叫什么？"}]},
        config=config
    )
    print(f"Agent: {response['messages'][-1].content}")




第一轮对话：
用户: 我叫李四

第二轮对话：
用户: 我叫什么？
Agent: 李四。


In [7]:
with SqliteSaver.from_conn_string(db_path) as checkpointer:
    agent = create_agent(
        model=model,
        tools=[],
        system_prompt="你是一个有帮助的助手。",
        checkpointer=checkpointer
    )

    # 使用相同的 thread_id
    config = {"configurable": {"thread_id": "persistent_session"}}

    print("\n第三轮对话（新进程，但 thread_id 相同）：")
    print("用户: 我之前说我叫什么？")
    response = agent.invoke(
        {"messages": [{"role": "user", "content": "我之前说我叫什么？"}]},
        config=config
    )
    print(f"Agent: {response['messages'][-1].content}")


第三轮对话（新进程，但 thread_id 相同）：
用户: 我之前说我叫什么？
Agent: 你之前说你叫李四。


In [8]:
db_path = "multi_user.sqlite"

with SqliteSaver.from_conn_string(db_path) as checkpointer:
    agent = create_agent(
        model=model,
        tools=[],
        system_prompt="你是一个有帮助的助手。",
        checkpointer=checkpointer
    )

    # 用户 A
    print("\n[用户 A 的对话]")
    config_a = {"configurable": {"thread_id": "user_alice"}}
    agent.invoke(
        {"messages": [{"role": "user", "content": "我是 Alice，我喜欢编程"}]},
        config_a
    )
    print("Alice: 我是 Alice，我喜欢编程")

    # 用户 B
    print("\n[用户 B 的对话]")
    config_b = {"configurable": {"thread_id": "user_bob"}}
    agent.invoke(
        {"messages": [{"role": "user", "content": "我是 Bob，我喜欢设计"}]},
        config_b
    )
    print("Bob: 我是 Bob，我喜欢设计")

    # 回到用户 A
    print("\n[用户 A 继续对话]")
    response_a = agent.invoke(
        {"messages": [{"role": "user", "content": "我喜欢什么？"}]},
        config_a
    )
    print(f"Alice: 我喜欢什么？")
    print(f"Agent: {response_a['messages'][-1].content}")

    # 回到用户 B
    print("\n[用户 B 继续对话]")
    response_b = agent.invoke(
        {"messages": [{"role": "user", "content": "我喜欢什么？"}]},
        config_b
    )
    print(f"Bob: 我喜欢什么？")
    print(f"Agent: {response_b['messages'][-1].content}")


    print(f"  - 数据库文件：{db_path}")


[用户 A 的对话]
Alice: 我是 Alice，我喜欢编程

[用户 B 的对话]
Bob: 我是 Bob，我喜欢设计

[用户 A 继续对话]
Alice: 我喜欢什么？
Agent: 你提到你喜欢编程，Alice！那太棒了！编程是一个非常有趣和动态的领域，随着技术的不断发展，它还在不断增长。除了编程之外，你还喜欢其他什么事情？你喜欢玩游戏、看电影、读书还是做其他事情？告诉我，我可以了解更多关于你的事情！

[用户 B 继续对话]
Bob: 我喜欢什么？
Agent: 我想你可能是在开玩笑，鲍勃！你一开始提到你喜欢设计。所以，我猜你喜欢设计……是这样吗？你想谈谈你最喜欢的设计项目或你最喜欢的设计风格吗？
  - 数据库文件：multi_user.sqlite


In [9]:
@tool
def get_order_status(order_id: str) -> str:
    """查询订单状态"""
    orders = {
        "12345": "已发货，预计明天送达",
        "67890": "配送中，今天下午送达"
    }
    return orders.get(order_id, "订单不存在")

In [10]:
db_path = "tools.sqlite"

with SqliteSaver.from_conn_string(db_path) as checkpointer:
    agent = create_agent(
        model=model,
        tools=[get_order_status],
        system_prompt="你是一个有帮助的助手。",
        checkpointer=checkpointer
    )

    config = {"configurable": {"thread_id": "customer_001"}}

    print("\n第一轮：查询订单")
    print("客户: 查询订单 12345 的状态")

    response1 = agent.invoke(
        {"messages": [{"role": "user", "content": "查询订单 12345 的状态"}]},
        config=config
    )
    print(f"Agent: {response1['messages'][-1].content}")


    print("\n第二轮：询问之前的查询结果")
    print("客户: 我的订单什么时候到？")
    response2 = agent.invoke(
        {"messages": [{"role": "user", "content": "我的订单什么时候到？"}]},
        config=config
    )
    print(f"Agent: {response2['messages'][-1].content}")



第一轮：查询订单
客户: 查询订单 12345 的状态
Agent: 订单 12345 的状态是：已发货，预计明天送达。如需更多信息，请提供更多订单详细信息。

第二轮：询问之前的查询结果
客户: 我的订单什么时候到？
Agent: 你的订单预计明天送达。


In [ ]:
import sqlite3

def view_database(db_path):
    if not os.path.exists(db_path):
        print(f"❌ 数据库文件不存在：{db_path}")

    print(f"\n{'='*70}")
    print(f"查看数据库：{os.path.basename(db_path)}")
    print(f"{'='*70}")

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 查看所有表
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    print(f"\n📋 数据库中的表：")
    for table in tables:
        print(f"  - {table[0]}")

    # 查看每个表的数据
    for table in tables:
        table_name = table[0]
        print(f"\n📊 表 '{table_name}' 的内容：")

        try:
            cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cursor.fetchone()[0]
            print(f"  记录数：{count}")

            # 显示前5条记录
            cursor.execute(f"SELECT * FROM {table_name} LIMIT 5")
            rows = cursor.fetchall()

            if rows:
                # 获取列名
                cursor.execute(f"PRAGMA table_info({table_name})")
                columns = [col[1] for col in cursor.fetchall()]
                print(f"  列：{', '.join(columns)}")

                print("\n  前5条记录：")
                for i, row in enumerate(rows, 1):
                    print(f"    [{i}] {row[:5]}...")  # 只显示前5个字段
            else:
                print("  （空表）")

        except sqlite3.Error as e:
            print(f"  ❌ 错误：{e}")

    conn.close()

In [14]:
db_files = [
    "checkpoints.sqlite",
    "multi_user.sqlite",
    "tools.sqlite",
]

print("\n" + "="*70)
print(" SQLite 数据库查看工具")
print("="*70)

for db_file in db_files:
    view_database(db_path)



 SQLite 数据库查看工具

查看数据库：tools.sqlite

📋 数据库中的表：
  - checkpoints
  - writes

📊 表 'checkpoints' 的内容：
  记录数：14
  列：thread_id, checkpoint_ns, checkpoint_id, parent_checkpoint_id, type, checkpoint, metadata

  前5条记录：
    [1] ('customer_001', '', '1f12db84-b4d7-6618-bfff-4fdfb7fc5822', None, 'msgpack')...
    [2] ('customer_001', '', '1f12db84-b4d9-68fa-8000-e4ded74e9968', '1f12db84-b4d7-6618-bfff-4fdfb7fc5822', 'msgpack')...
    [3] ('customer_001', '', '1f12db84-b730-67ac-8001-4ec741e5ec15', '1f12db84-b4d9-68fa-8000-e4ded74e9968', 'msgpack')...
    [4] ('customer_001', '', '1f12db84-b781-6e9a-8002-fd78fd005850', '1f12db84-b730-67ac-8001-4ec741e5ec15', 'msgpack')...
    [5] ('customer_001', '', '1f12db84-ba64-6018-8003-1573f64861ff', '1f12db84-b781-6e9a-8002-fd78fd005850', 'msgpack')...

📊 表 'writes' 的内容：
  记录数：22
  列：thread_id, checkpoint_ns, checkpoint_id, task_id, idx, channel, type, value

  前5条记录：
    [1] ('customer_001', '', '1f12db84-b4d7-6618-bfff-4fdfb7fc5822', '415dba38-3cfd-b65b-